# Driving complaints, collisions and vehicle stops by police beat

**A5.** Across the 14 Berkeley police beats in 2024–2025, do beats with more reckless-driving calls also record more collisions and vehicle stops? We compare counts in the same two calendar years using Spearman rank correlation, giving each beat equal weight. The counts describe recorded activity and depend on traffic volume, reporting and enforcement.

Read the processed calls, stops, collisions and address lookup, plus the shared Police Beats polygons. The output `data/processed/a5_driving_by_beat.csv` contains the primary counts and the alternative collision counts described below.

Calls inherit the sector of the median crime-case point for their normalized block address. See the [address lookup codebook](../data/codebooks/address_sectors.md) for match statuses and the shared BPD beat layer. Of 1,091 reckless-driving calls, 1,051 (96.3%) matched a beat; 31 had unmatched addresses, eight fell outside and one was ambiguous.

The boundary sensitivity counts are saved in `results/06_boundary_summary.csv`.

In [1]:
import numpy as np
import pandas as pd

from bpdtools.config import RECKLESS_CALL_TYPES
from bpdtools.paths import repo_path
from bpdtools.sectors import (
    assign_sectors,
    attach_address_sectors,
    load_sector_features,
)
from bpdtools.traffic import load_collisions, load_stops

ANALYSIS_YEARS = (2024, 2025)
WEB_MERCATOR_RADIUS_M = 6_378_137

processed = repo_path("data", "processed")
features = load_sector_features()
stops = load_stops()
collisions = load_collisions()
missing_dates = [stops.StopDate.isna().sum(), collisions.Date.isna().sum()]
# Use the same two calendar years for all three sources.
stops = stops.loc[stops.StopDate.dt.year.isin(ANALYSIS_YEARS)]
collisions = collisions.loc[collisions.Date.dt.year.isin(ANALYSIS_YEARS)]
# The collision file stores Web Mercator metres. Convert to the map's degrees.
lon = np.degrees(collisions.x / WEB_MERCATOR_RADIUS_M)
lat = np.degrees(np.arctan(np.sinh(collisions.y / WEB_MERCATOR_RADIUS_M)))
stop_labels = pd.Series(assign_sectors(stops.Longitude, stops.Latitude, features))
collision_labels = pd.Series(assign_sectors(lon, lat, features), index=collisions.index)
calls = pd.read_csv(processed / "calls_2021_2025.csv.gz", parse_dates=["date"])
# Keep calls in the shared reckless-driving category during the study years.
selected = calls["Call Type"].isin(RECKLESS_CALL_TYPES) & calls.date.dt.year.isin(
    ANALYSIS_YEARS
)
reckless = attach_address_sectors(
    calls.loc[selected], pd.read_csv(processed / "address_sectors.csv")
)
beats = sorted(int(f["properties"]["Sector"]) for f in features)
table = pd.DataFrame(index=pd.Index(beats, name="beat"))
# Keep all 14 beats in the comparison, including beats with zero events.
for column, labels in {
    "vehicle_stops": stop_labels,
    "collisions": collision_labels,
    "reckless_calls": reckless.Sector,
}.items():
    table[column] = (
        pd.to_numeric(labels, errors="coerce")
        .value_counts()
        .reindex(beats, fill_value=0)
    )
print("Missing stop and collision dates in full snapshots:", missing_dates)
print("Spatial exclusions for dated stops and collisions:")
display(
    pd.DataFrame(
        {
            "stops": stop_labels.value_counts(),
            "collisions": collision_labels.value_counts(),
        }
    )
    .reindex(["missing", "outside", "ambiguous"])
    .fillna(0)
)
display(reckless.sector_status.value_counts())

Missing stop and collision dates in full snapshots: [np.int64(0), np.int64(101)]
Spatial exclusions for dated stops and collisions:


,stops,collisions
missing,41.0,0.0
outside,365.0,39.0
ambiguous,0.0,3.0


sector_status
assigned     1051
unmatched      31
outside         8
ambiguous       1
Name: count, dtype: int64

## Boundary sensitivity

For each assigned collision, find all beat boundaries within one centimetre of its point. Distances use a local tangent-plane approximation around Berkeley. If there is exactly one alternative beat and the original boundary also falls within the threshold, move the collision to that alternative. Keep the same events, calls and stops in both comparisons.

In [2]:
EARTH_MEAN_RADIUS_M = 6_371_000
BERKELEY_LATITUDE_DEG = 37.87
EDGE_TOLERANCE_M = 0.01

# Convert degrees to local metres, then measure the shortest point-to-segment distance.
metres_per_degree = EARTH_MEAN_RADIUS_M * np.pi / 180
scale = metres_per_degree * np.array([np.cos(np.radians(BERKELEY_LATITUDE_DEG)), 1.0])
points = np.column_stack([lon, lat]) * scale
near_edge = pd.DataFrame(False, index=collisions.index, columns=beats)
for feature in features:
    beat = int(feature["properties"]["Sector"])
    geometry = feature["geometry"]
    parts = geometry["coordinates"]
    if geometry["type"] == "Polygon":
        parts = [parts]
    close = np.zeros(len(points), dtype=bool)
    for polygon in parts:
        for ring in polygon:
            vertices = np.asarray(ring)[:, :2] * scale
            for start, end in zip(vertices[:-1], vertices[1:]):
                delta = end - start
                length_squared = delta @ delta
                if length_squared == 0:
                    continue
                # Only evaluate points inside the segment's expanded bounding box.
                candidate = np.flatnonzero(
                    (
                        (points >= np.minimum(start, end) - EDGE_TOLERANCE_M)
                        & (points <= np.maximum(start, end) + EDGE_TOLERANCE_M)
                    ).all(axis=1)
                )
                offsets = points[candidate] - start
                fraction = np.clip((offsets @ delta) / length_squared, 0, 1)
                residual = offsets - fraction[:, None] * delta
                close[candidate] |= np.linalg.norm(residual, axis=1) <= EDGE_TOLERANCE_M
    near_edge[beat] = close
alternate = pd.to_numeric(collision_labels, errors="coerce").copy()
switched = 0
# With exactly two nearby beats, an assigned collision has one alternative.
two_boundaries = near_edge.sum(axis=1).eq(2)
for index, row in near_edge.loc[two_boundaries].iterrows():
    pair = row.index[row]
    if alternate.at[index] in pair:
        alternate.at[index] = next(beat for beat in pair if beat != alternate.at[index])
        switched += 1
table["alternate_collisions"] = alternate.value_counts().reindex(beats, fill_value=0)
assert table.collisions.sum() == table.alternate_collisions.sum()
table.to_csv(processed / "a5_driving_by_beat.csv")
display(table)
print("Reassigned collisions:", switched)
changed_ranks = table.collisions.rank().ne(table.alternate_collisions.rank()).sum()
print("Changed beat ranks:", changed_ranks)
# Save the three counts quoted in the report from this same reassignment.
boundary_summary = pd.DataFrame(
    {
        "collisions_near_two_boundaries": [two_boundaries.sum()],
        "reassigned_collisions": [switched],
        "beats_with_changed_rank": [changed_ranks],
    }
)
boundary_summary.to_csv(repo_path("results", "06_boundary_summary.csv"), index=False)
display(
    table.corr(method="spearman").loc[
        "reckless_calls", ["collisions", "vehicle_stops", "alternate_collisions"]
    ]
)

,vehicle_stops,collisions,reckless_calls,alternate_collisions
beat,,,,
1,73,128,64,128
2,340,149,85,149
3,814,122,82,123
4,618,76,49,88
5,560,149,124,139
6,548,159,76,156
7,419,145,107,145
8,221,94,78,90
9,458,130,70,135


Reassigned collisions: 58
Changed beat ranks: 10


collisions              0.753304
vehicle_stops           0.191419
alternate_collisions    0.730176
Name: reckless_calls, dtype: float64

Across 14 beats, reckless-driving calls tracked recorded collisions more closely (Spearman ρ = 0.753) than vehicle stops (ρ = 0.191). Under the one-centimetre rule, 59 collisions lay near two boundaries. One already had ambiguous membership; the other 58 moved to their alternative beat. Ten beats changed collision rank (average ranks for ties), while the calls–collisions correlation remained positive at 0.730. These area-level counts do not identify whether a particular complaint preceded a collision or whether stops prevented one.